# KAN dentro de FitSNAP — Vast.ai

**Proyecto:** potenciales interatómicos reactivos · Li-F · **Autor:** Brian Jara, UNI

Este notebook usa **el mismo flujo que el equipo**: FitSNAP lee los JSON, llama a LAMMPS para
los descriptores, entrena y escribe las métricas. Lo único que cambia es la **última capa de
la red**, que pasa de ser un `Linear` a ser un bloque KAN.

Todo corre en una instancia de Vast.ai.

---

## Por qué usar FitSNAP y no un entrenamiento propio

| Razón | Detalle |
|---|---|
| **Comparabilidad** | Tus resultados salen en el mismo `metrics.dat` que los de Luis, con los mismos pesos de energía y fuerza, el mismo potencial de referencia y la misma normalización por átomo |
| **Menos que reimplementar** | FitSNAP ya hace el andamiaje. Si lo reescribes, cualquier diferencia silenciosa cambia tus números |
| **El cambio es chico** | Toda la red se construye en una función de 35 líneas, `create_torch_network` |

## Dónde NO hay solapamiento con el equipo

| Persona | Qué modifica |
|---|---|
| Arbués | El **scraper** — la parte que lee los datos y los mete en memoria |
| Jorge | Nada del código: entrena y compara rendimiento |
| **Tú** | La **red** — `lib/neural_networks/pytorch.py` |

Son archivos distintos. Y aquí no se edita ningún archivo: la red se sustituye en memoria al
vuelo, así que la instalación de FitSNAP queda intacta.

## 0. Procedencia de cada decisión

| Decisión | Valor | De dónde sale |
|---|---|---|
| KAN en el bloque de salida | última capa | Wang, Yu, Zhong & Xiang (2024), *J. Mater. Inf.* 4:28 — gana en los 4 modelos que probaron |
| Base principal | gaussiana | Wang et al. (2024): nunca perdió contra el MLP |
| Base de control | B-spline, G=5, k=3 | Liu et al. (2024), arXiv:2404.19756 — valores por defecto del KAN original |
| Base alternativa | Chebyshev, grado 5 | Mahmoud et al. (2025), *IEEE Access* |
| Base descartada | Fourier | Wang et al. (2024): la única que perdió contra el MLP |
| Igualar parámetros | en los dos sentidos | Yu, Yu & Wang (2024), arXiv:2407.16674 |
| Semillas | 5 | Wang et al. usaron 3 y una divergió |
| Tiempo de reloj además de FLOPs | obligatorio | Raffel et al. (2025) AAAI — FlashKAT: 123× más lento con FLOPs iguales |
| Activación del tronco | **Softplus** | Es la que usa FitSNAP en `create_torch_network`. No la cambies o dejas de ser comparable |
| Potencial de juguete | Lennard-Jones | Nagai & Okumura (2024), arXiv:2407.17774 |
| Continuidad de la derivada | criterio de selección | `[DECISIÓN PROPIA]` La fuerza es −dE/dr; una base C⁰ da fuerzas con saltos |

## Cómo se usa este notebook

**Se corre de arriba abajo.** Ninguna celda depende de otra posterior y ninguna hay que
ejecutarla dos veces. Si hace falta reiniciar el kernel, se reinicia y se empieza desde
arriba.

**El código no está aquí.** Las capas KAN, los parches de compatibilidad, la configuración
de FitSNAP y el cambio de arquitectura viven en `kan_lib.py`, dentro de `MARCO_KAN/`. Este
notebook los importa y los usa; no los define. Antes se generaba esa librería en cada
corrida volcando funciones desde la memoria de la sesión, y el volcado no arrastraba los
globales de los que dependían: tres pilotos murieron por un nombre que faltaba, a 8–15
minutos de GPU cada uno. Ahora es un archivo, y hay una comprobación que lo revisa en un
segundo antes de lanzar nada.

| Archivo de `MARCO_KAN/` | Qué hace |
|---|---|
| `kan_lib.py` | El módulo: capas, parches, configuración, cambio de arquitectura |
| `verificar_lib.py` | Lista nombres usados y no definidos. Un segundo |
| `prueba_variantes.py` | Compara los cuatro modos de rango sin GPU ni FitSNAP |
| `barrida.py` | Calibración de lote, de tasa y de modo de rango |
| `trabajador.py` | Un proceso, un lote de corridas |
| `lanzar.py` | Reparte las corridas entre las tarjetas |
| `monitor_fases.py` | Mide recursos separando descriptores de entrenamiento |
| `sonda_gradientes.py` | Mide el gradiente que llega a cada capa dentro del pipeline |
| `mae.py` | **El criterio.** Errores físicos desde las predicciones |
| `tabla70.py` | Junta las corridas largas y marca las inservibles |
| `comparar.py` | Envoltorio de `compare_runs.py`: tabla contra baseline y parity plots |

### El orden

Secciones 1 a 10: instalación, verificación y maquinaria. Sección 11: calibración.
**Sección 12: el presupuesto**, que es la que decide si la campaña puede medir algo.
Sección 13: el criterio de medida. Secciones 14 a 17: condiciones, lanzamiento y envío
al clúster.

En una máquina nueva se corre todo. En una campaña de producción basta con 1 a 3 para
verificar, luego 12, 13 y 17.

## 1. Qué máquina te tocó

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!echo "--- RAM (GB) ---" && free -g | head -2
!echo "--- nucleos ---" && nproc
!echo "--- disco ---"   && df -h ${SCRATCH:-/tmp} ${WORK:-/tmp} /workspace /root . 2>/dev/null | tail -n +2 | sort -u

## 2. Instalar FitSNAP y LAMMPS

Desde conda-forge. El script `montaje_vastai.sh` ya hace esto; la celda está por si prefieres
hacerlo desde aquí. Tras correrla hay que **cambiar el kernel a "Python (fitsnap)"**.

In [ ]:
%%bash
BASE=$(for d in "${KAN_RAIZ:-}" "${SCRATCH:-}" "${WORK:-}" /workspace /root .; do
  [ -z "$d" ] && continue; [ -d "$d" ] && echo "$(df --output=avail -k $d|tail -1) $d"; done | sort -rn | head -1 | awk '{print $2}')
if ! command -v conda >/dev/null 2>&1; then
  curl -fsSL -o /tmp/mf.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
  bash /tmp/mf.sh -b -p "$BASE/miniforge3"
fi
source "$BASE/miniforge3/etc/profile.d/conda.sh"
conda env list | grep -q "^fitsnap " || \
  mamba create -y -n fitsnap python=3.11 lammps fitsnap3 pytorch-gpu \
        numpy scipy pandas matplotlib scikit-learn mpi4py openmpi \
        jupyterlab ipykernel -c conda-forge
conda activate fitsnap
python -m ipykernel install --user --name fitsnap --display-name "Python (fitsnap)"

In [ ]:
# Verificacion. Correr YA con el kernel "Python (fitsnap)".
import sys, os, shutil
from pathlib import Path

estado = {}
try:
    import lammps
    L = lammps.lammps(cmdargs=["-log", "none", "-screen", "none"])
    pk = L.installed_packages
    estado["LAMMPS"] = "OK"
    estado["ML-SNAP"] = "OK" if "ML-SNAP" in pk else "FALTA <-- sin esto no hay descriptores"
    estado["ML-IAP"] = "OK" if "ML-IAP" in pk else "falta (solo para desplegar)"
except Exception as e:
    estado["LAMMPS"] = f"FALLO: {e}"
for m in ("fitsnap3lib", "torch", "numpy", "pandas"):
    try:
        __import__(m); estado[m] = "OK"
    except Exception as e:
        estado[m] = f"FALLO: {e}"
for k, v in estado.items():
    print(f"  {k:12s} {v}")

_c = [Path(p) for p in (os.environ.get("SCRATCH", ""), os.environ.get("WORK", ""),
                        "/workspace", "/root", ".") if p and Path(p).exists()]
RAIZ = (Path(os.environ["KAN_RAIZ"]) if os.environ.get("KAN_RAIZ")
        else max(_c, key=lambda p: shutil.disk_usage(p).free) / "kan_snap")
for s in ("resultados", "checkpoints"):
    (RAIZ / s).mkdir(parents=True, exist_ok=True)
print(f"\\n  carpeta de trabajo: {RAIZ}")

## 3. El módulo y los parches

Una sola celda deja todo listo: importa `kan_lib`, aplica los siete parches de
compatibilidad y recién entonces importa FitSNAP. **El orden importa**: los parches tienen
que estar puestos antes del primer `import fitsnap3lib`.

La celda termina con tres comprobaciones contra la versión de FitSNAP realmente instalada.

1. Que `evaluate_configs` sustituye el modelo por `model_best`. Es la causa de que una
   segunda llamada a `perform_fit` sobre el mismo objeto no entrene, y lo que obliga a pasar
   por `cambiar_arquitectura`.
2. Que FitSNAP **lee** la clave `dtype_setting`.
3. Que `ajustes_fitsnap` **no la envía**.

Las dos últimas parecen redundantes y no lo son. Que la clave se lea no significa que se
admita: `_check_section` mantiene una lista de claves válidas por sección y `dtype_setting`
no está en ella, de modo que enviarla aborta la construcción del objeto con
`Found unmatched variable in PYTORCH section of input`. Todo corre en float32.

In [ ]:
import sys, os, shutil
from pathlib import Path

# La raiz se resuelve una sola vez y manda en todo el notebook.
#   1. KAN_RAIZ si esta puesta en el entorno (es lo que usa un cluster)
#   2. la que calculo la seccion 1 por espacio libre
#   3. el primer sitio escribible de la lista, por espacio libre
if os.environ.get("KAN_RAIZ"):
    RAIZ = Path(os.environ["KAN_RAIZ"])
elif "RAIZ" in dir():
    RAIZ = Path(RAIZ)
else:
    _cand = [Path(p) for p in (os.environ.get("SCRATCH", ""),
                               os.environ.get("WORK", ""),
                               "/workspace", str(Path.home()), ".")
             if p and Path(p).exists()]
    RAIZ = max(_cand, key=lambda p: shutil.disk_usage(p).free) / "kan_snap"
MARCO = RAIZ / "MARCO_KAN"
sys.path.insert(0, str(MARCO))

from kan_lib import *
from kan_lib import (aplicar_todos_los_parches, ajustes_fitsnap,
                     cambiar_arquitectura, resumen_kan, carpeta_datos)

aplicar_todos_los_parches()
from fitsnap3lib.fitsnap import FitSnap

print("kan_lib cargado. RAIZ =", RAIZ)
print("configuracion efectiva del bloque KAN:")
print("   ", resumen_kan())
print()

import inspect
import fitsnap3lib.solvers.pytorch as _P
import fitsnap3lib.io.sections.solver_sections.pytorch as _S

print("1. evaluate_configs sustituye el modelo por model_best:",
      "self.model = self.model_best" in inspect.getsource(_P))
print("2. FitSNAP LEE la clave dtype_setting:                 ",
      "dtype_setting" in inspect.getsource(_S))
_cfg_prueba = ajustes_fitsnap(carpeta_datos(RAIZ), grupos=["LiF_64_kjpaw"])
print("3. ajustes_fitsnap NO la envia:                        ",
      "dtype_setting" not in _cfg_prueba["PYTORCH"])
print()
print("Si la 1 dice False, el diagnostico no aplica a esta instalacion.")
print("Si la 3 dice False, la construccion del objeto va a abortar.")

## 4. Las capas KAN

Una capa KAN reemplaza cada peso escalar por una **función univariada aprendible**:

$$y_j = \sum_{i} \varphi_{ij}(x_i), \qquad \varphi_{ij}(x) = \sum_{k=1}^{K} w_{ijk}\, b_k(x)$$

Las $b_k$ son formas fijas —las campanas del gráfico— y lo que se aprende son las alturas
$w_{ijk}$. **Parámetros: `n_in × n_out × K`** frente a `n_in × n_out` del `Linear`.

Las tres bases tienen la misma interfaz, así que se intercambian sin tocar nada más.

In [ ]:
# Las tres bases disponibles, y cuantos parametros anade cada una.
print("bases disponibles:", list(BASES))
print()
print(f"{'base':12s} {'capa 64->1':>12s} {'parametros':>12s}")
print("-" * 38)
import torch.nn as nn
_lineal = nn.Linear(64, 1)
print(f"{'linear':12s} {'Linear':>12s} {sum(p.numel() for p in _lineal.parameters()):12,}")
for _nombre, _clase in BASES.items():
    _kw = dict(gaussiana=dict(K=8), bspline=dict(G=5, k=3),
               chebyshev=dict(grado=5))[_nombre]
    _c = _clase(64, 1, **_kw)
    print(f"{_nombre:12s} {_clase.__name__:>12s} "
          f"{sum(p.numel() for p in _c.parameters()):12,}")

## 5. La red de FitSNAP, con el bloque de salida intercambiable

La función original de FitSNAP, en `fitsnap3lib/lib/neural_networks/pytorch.py`, arma esto
para `layer_sizes = [440, 256, 128, 64, 64, 1]`:

```
Linear(440→440)   ← capa de estandarización
Linear(440→256) → Softplus
Linear(256→128) → Softplus
Linear(128→64)  → Softplus
Linear(64→64)   → Softplus
Linear(64→1)    ← el bloque de salida
```

Se verificó contra el modelo entrenado de Luis (`FitTorch_Pytorch.pt`): las capas están en
los índices 0, 1, 3, 5, 7 y 9, con activaciones en 2, 4, 6 y 8. Coincide exactamente.

**Nuestra versión reproduce esa red idéntica y solo cambia la última capa.**

In [ ]:
# La red completa, con el bloque de salida en "linear": debe coincidir
# exactamente con la que arma FitSNAP.
_LS = [440, 256, 128, 64, 64, 1]
_base = crear_red(_LS, "linear")
print("estructura de la linea base:")
for _i, _c in enumerate(_base):
    print(f"  [{_i}] {_c}")
print()
print(f"parametros: {n_params(_base):,}")

## 6. El parche: sustituir la red sin tocar archivos

FitSNAP importa `create_torch_network` en tres sitios distintos. Se reemplazan los tres **en
memoria**, así que la instalación queda intacta y basta reiniciar el kernel para volver atrás.

### Compatibilidad: FitSNAP contra versiones nuevas de PyTorch y Python

FitSNAP 3.1.0.4 es de 2024; conda-forge instala LAMMPS, PyTorch, Python y setuptools al
día. Siete llamadas cambiaron desde entonces. Ninguna cambió de *comportamiento*: son
renombres, argumentos retirados y funciones movidas.

| # | Qué cambió | Quién lo cambió | Por qué el arreglo no altera nada |
|---|---|---|---|
| 1 | `ReduceLROnPlateau(verbose=True)` | PyTorch | `verbose` solo imprimía un mensaje |
| 2 | `shuffle(lista, random)` | Python 3.11 | ese 2º argumento *era* la fuente por defecto |
| 3 | `create_atoms(id=, type=)` → `(atomid=, atype=)` | LAMMPS | renombre puro; mismo orden, mismo cuerpo |
| 4 | `strtobool` devuelve `True` y ya no `1` | setuptools | se restaura el entero que FitSNAP espera |
| 5 | `extract_atom_iarray` / `_darray` eliminados | LAMMPS | se reponen con el código exacto de 2024.8 |
| 6 | `index_add_` rechaza tensores sin dimensiones | PyTorch | se añade la dimensión al entrar y se quita al salir |
| 7 | Triton compila un módulo en C y no halla `cuda.h` | PyTorch 2.13 | se le indica a `gcc` dónde está, vía `CPATH` |

Las siete se corrigen envolviendo o reponiendo la función original. **No se edita ningún
archivo**, igual que el parche de la red: reiniciando el kernel todo vuelve atrás.

**Sobre el 4, que es el menos obvio.** FitSNAP hace `from distutils.util import strtobool`
para leer las banderas de la configuración. En la versión original esa función devolvía el
entero `1` o `0`; setuptools, que hoy suplanta a `distutils`, devuelve `True` o `False`.
FitSNAP luego mete ese valor tal cual dentro de un comando de LAMMPS, y sale
`bzeroflag True` — que LAMMPS rechaza, porque ahí espera un número. Devolver enteros
restaura exactamente el comportamiento contra el que FitSNAP fue escrito, y no rompe nada
más: en Python `if 1:` y `if True:` son lo mismo.

**Cobertura.** Se revisaron uno por uno los métodos de LAMMPS que FitSNAP 3.1.0.4 invoca:
los nueve de la interfaz principal (`close`, `command`, `commands_string`, `create_atoms`,
`extract_atom`, `extract_compute`, `find_pair_neighlist`, `get_natoms`, `get_thermo`) y los
cinco del envoltorio numpy (`extract_atom`, `extract_atom_darray`, `extract_atom_iarray`,
`extract_compute`, `get_neighlist`). Los tres afectados están en la tabla.

Es el mismo tipo de problema de versiones que reportó Jorge en la reunión.

In [ ]:
# Prueba en seco: que ultima capa produciria FitSNAP con el parche puesto.
from fitsnap3lib.lib.neural_networks.pytorch import create_torch_network

parchear_red("gaussiana", K=8)
from fitsnap3lib.lib.neural_networks.pytorch import create_torch_network as _f
print("con el parche, ultima capa:", _f([440, 64, 1])[-1].__class__.__name__)

restaurar_red()
from fitsnap3lib.lib.neural_networks.pytorch import create_torch_network as _g
print("restaurado,  ultima capa:", _g([440, 64, 1])[-1].__class__.__name__)

## 7. Configuración de FitSNAP

Los mismos ajustes del `.in` del equipo, pasados como diccionario para poder correr en modo
librería. **No cambies los pesos ni los descriptores**: son lo que hace tus resultados
comparables con los suyos.

In [ ]:
# La configuracion, tal como la recibe FitSNAP. Los valores compartidos por las
# 8 condiciones son variables del modulo y se pueden pisar por entorno.
print(resumen_kan())
print()

_cfg = ajustes_fitsnap(carpeta_datos(RAIZ), grupos=["LiF_64_kjpaw"])
print("secciones:", ", ".join(_cfg))
print()
for _k, _v in _cfg["PYTORCH"].items():
    print(f"  PYTORCH.{_k:20s} {_v}")
print()
print("dtype_setting no aparece arriba, y es correcto que no aparezca: esta")
print("version de FitSNAP la rechaza. Todo corre en float32.")

### La estructura de carpetas que exige FitSNAP

FitSNAP construye la ruta como `dataPath + nombre_del_grupo`. Por eso `dataPath` tiene que
apuntar a la carpeta **padre**, con una subcarpeta por grupo:

```
kan_snap/JSON/            <-- esto es dataPath
├── BCC_54_kjpaw/
│   ├── output_0.json
│   └── ...
├── LiF_64_kjpaw/
└── Special/
```

Es la misma convencion del `.in` del equipo, donde `dataPath = ../LiF_kjpaw_chem/JSON`.

In [ ]:
RUTA_DATOS = carpeta_datos(RAIZ)   # carpeta padre, NO la de un grupo concreto
#   dataset/ si el codigo vino del repositorio; JSON/ si se convirtio aqui

def revisar_estructura(raiz_datos=RUTA_DATOS, grupos=None):
    """Comprueba que cada grupo tenga su carpeta con JSON dentro."""
    raiz_datos = Path(raiz_datos)
    if not raiz_datos.is_dir():
        print(f"NO EXISTE: {raiz_datos}")
        return {}
    conteo = {}
    for d in sorted(raiz_datos.iterdir()):
        if d.is_dir():
            conteo[d.name] = len(list(d.glob("*.json")))
    for g, n in conteo.items():
        marca = "" if n else "   <-- VACIA"
        print(f"  {g:24s} {n:6d}{marca}")
    print(f"  {'TOTAL':24s} {sum(conteo.values()):6d}")
    if grupos:
        faltan = [g for g in grupos if g not in conteo]
        if faltan:
            print(f"\n  FALTAN carpetas para: {faltan}")
    return conteo

_ = revisar_estructura()

## 8. Una corrida de FitSNAP

El flujo de la librería es de cuatro pasos:

```python
fs.scrape_configs()    # lee los JSON
fs.process_configs()   # calcula descriptores con LAMMPS   <- el paso caro
fs.perform_fit()       # entrena
fs.write_output()      # escribe metrics.dat, perconfig, peratom
```

**Clave para que todo entre en una sesión:** `process_configs()` se corre **una sola vez**.
Los descriptores no dependen de la arquitectura de la red, así que los 8 experimentos se
ajustan sobre la misma matriz.

### La prueba corta

Una condición, un grupo, 3 épocas. Sirve para comprobar que la tubería completa funciona
antes de gastar horas.

`LiF_64_kjpaw` tiene los dos elementos (32 Li + 32 F), así que no aparecen descriptores
constantes. Un grupo de litio puro daría `1/0` en la estandarización.

Esta celda es válida porque es **una sola** llamada a `perform_fit` sobre un objeto recién
construido. Encadenar varias sobre el mismo objeto sin pasar por `cambiar_arquitectura` no
entrena: el optimizador queda apuntando a los tensores del modelo anterior.

**Tres épocas no dicen si una arquitectura aprende.** Dicen si el andamiaje funciona. Para
lo primero hacen falta los pasos de la sección 14.

In [ ]:
r = una_corrida(
    ruta_json      = carpeta_datos(RAIZ),    # la carpeta PADRE, no la de un grupo
    salida_kan     = "gaussiana",
    kw_kan         = dict(K=8),
    semilla        = 1111,
    carpeta_salida = RAIZ / "prueba_corta",
    epocas         = 3,                      # 3 en vez de 70: solo es la prueba
    grupos         = ["LiF_64_kjpaw"],       # 32 Li + 32 F
)

print()
print("=" * 62)
for k in ("seg_descriptores", "seg_ajuste"):
    print(f"  {k:20s} {r[k]/60:8.1f} min")
print(f"  {'descriptores muertos':20s} {r['descriptores_muertos']:8d}")
print(f"  {'metrics.dat':20s} {r['metrics']}")
print("=" * 62)

## 9. La rejilla: 8 condiciones × 5 semillas

Las filas 7 y 8 son el control de capacidad:

- **7** responde: si le quito parámetros al KAN hasta igualar al MLP, ¿sigue ganando?
- **8** responde: si le doy al MLP los parámetros del KAN, ¿alcanza al KAN?

La 8 no está en ningún paper de esta literatura.

**Resultado adelantado:** el bloque KAN añade 448 parámetros sobre 352 313, un **0.127 %**.
En esta configuración la capacidad extra es despreciable por construcción, así que las filas
7 y 8 confirman formalmente algo que ya se ve en la tabla de parámetros. Es un control más
fuerte que el de Wang et al., que no reportan el conteo.

In [ ]:
# Las 8 condiciones, con su conteo de parametros.
print(f"{'id':18s} {'salida':11s} {'parametros':>12s}")
print("-" * 44)
for _cid, _capas, _sal, _kw in CONDICIONES:
    _ls = [440] + [int(x) for x in _capas.split()[1:]]
    print(f"{_cid:18s} {_sal:11s} {n_params(crear_red(_ls, _sal, **_kw)):12,}")
print()
print(f"semillas: {SEMILLAS}")
print(f"corridas totales: {len(CONDICIONES) * len(SEMILLAS)}")

## 10. Recoger los resultados

`metrics.dat` de FitSNAP trae MAE y RMSE de energía y fuerza, por grupo y separando
entrenamiento de prueba. **La energía viene en eV por átomo** — FitSNAP divide por el número
de átomos antes de reportar; verificado siguiendo el número desde el `.out` de Quantum
ESPRESSO hasta `perconfig.dat`.

### Anclas externas para situar tus números

Publicadas, independientes del grupo. Dataset de carbono, RMSE.

| Potencial | Energía (meV/át.) | Fuerza (meV/Å) |
|---|---:|---:|
| GAP | 46 | 1100 |
| DP | 44 | 800 |
| MTP | 35 | 630 |
| MACE | 8.0 | 307 |
| Exactitud química | 43 meV | — |

---

## 11. Calibración: tamaño de lote, tasa y modo de rango

Tres parámetros compartidos por las ocho condiciones. Moverlos no afecta la comparación
entre arquitecturas; solo rompe la comparabilidad de los números absolutos contra el
póster del grupo.

**La maquinaria tiene tres resguardos, y los tres hacen falta.** `barrida.py` calcula los
descriptores una vez y los reutiliza entre valores; pasa por `cambiar_arquitectura`, que
reconstruye el optimizador sobre el modelo vigente y borra `model_best`; y comprueba por
identidad, antes de cada ajuste, que optimizador y modelo comparten los mismos tensores.
Después mide si los pesos se movieron: una fila con `delta = 0` se marca como no entrenada
y su pérdida no significa nada.

**Y la utilización de GPU se mide por fase.** Promediarla sobre toda la corrida no dice
nada: de cada 456 segundos, 452 son descriptores en CPU.

### Lo que una barrida corta puede y no puede decidir

Una barrida de pocas épocas sirve para **descartar** valores que divergen y para medir
segundos por época y memoria. No sirve para **elegir** entre valores que no divergen,
porque penaliza por construcción a las tasas pensadas para corridas largas: la
configuración de referencia del grupo usa `5e-6` durante 70 épocas y alcanza 1.108e-02
eV/Å, y a 8 épocas esa misma tasa parece plana.

Y hay que barrer **sobre la arquitectura que se va a usar**. El valor por omisión de
`--arquitectura` en `barrida.py` es `linear`; una tasa calibrada sobre el MLP no es
necesariamente la del KAN, porque el modo de gestión del rango cambia la escala del
gradiente que llega a la cabecera.

### Preparar los descriptores una sola vez

Este es el paso caro (LAMMPS). Se hace una vez y las cuatro tasas lo reutilizan.

Se usa el **MLP base** a propósito: la pregunta de Selis es sobre el learning rate, así que el
bloque de salida se deja como el del grupo.

### Primero el lote

**El orden importa.** Lote y tasa interactúan: con lotes más grandes el gradiente tiene
menos ruido y admite pasos mayores. Decidir el lote después de barrer las tasas invalida la
barrida de tasas.

Dos medidas de VRAM, y hay que mirar la segunda:

| Medida | Qué es |
|---|---|
| `nvidia-smi` | incluye la caché del asignador de PyTorch: **sobrestima** |
| `torch.cuda.max_memory_allocated()` | lo que el modelo realmente pidió: **el número bueno** |

**El criterio de elección no es la velocidad.** Un lote mayor da menos actualizaciones de
pesos por época, y en este problema el número de actualizaciones es la variable que manda
—ver la sección 14—. El lote se elige dentro del presupuesto de pasos, no antes de fijarlo.

FitSNAP llama a `create_datasets` en cada `perform_fit` y lee `batch_size` de la
configuración en vivo, así que basta cambiarlo entre corridas.

In [ ]:
# ~10 min: descriptores una vez + 5 tamanos.
# Sin --silencioso: en ese modo el monitor descarta la salida del hijo, traceback
# incluido, y un fallo deja un registro que solo trae el pie del monitor.
!cd {MARCO} && python monitor_fases.py \
    --csv {RAIZ}/barrida_lote.csv -- \
    python barrida.py --que lote --raiz {RAIZ} --grupo LiF_64_kjpaw --epocas 8

### La tasa, sobre cada arquitectura

Dos barridas, una por arquitectura, porque sus óptimos no tienen por qué coincidir.

| Lo que veas | Qué significa | Qué hacer |
|---|---|---|
| alguna tasa con pérdida inicial > 1e6 | divergió en el primer paso | descartarla, y todas las mayores |
| todas con `delta = 0` | no entrenó ninguna: revisar el acoplamiento del optimizador | parar |
| diferencias pequeñas entre tasas vecinas | la barrida es demasiado corta para distinguirlas | no elegir aquí; decidir con una corrida larga |

In [ ]:
!cd {MARCO} && python monitor_fases.py \
    --csv {RAIZ}/barrida_lr_mlp.csv -- \
    python barrida.py --que lr --raiz {RAIZ} --grupo LiF_64_kjpaw \
    --arquitectura linear --epocas 8

!cd {MARCO} && python monitor_fases.py \
    --csv {RAIZ}/barrida_lr_kan.csv -- \
    python barrida.py --que lr --raiz {RAIZ} --grupo LiF_64_kjpaw \
    --arquitectura gaussiana --epocas 8

### El modo de gestión del rango

La capa que precede al bloque KAN es un `Softplus`, siempre positivo, mientras que las bases
se reparten en `(-2, 2)`. Hay cuatro formas de tratarlo y **no son equivalentes**:

| modo | qué hace |
|---|---|
| `no` | el control: no se toca nada. **Es el valor por omisión** |
| `canal` | estandariza por canal con las estadísticas del primer lote, congeladas |
| `layernorm` | normaliza por muestra a lo ancho de los canales, recalculado en cada paso |
| `cuantiles` | mueve la rejilla a los cuantiles de los datos, una vez |

Llevar los datos a la rejilla con media y desviación, y mover la rejilla a los datos con esa
misma media y desviación, son el **mismo modelo reparametrizado** —comprobado, diferencia
3.6e-07—. Lo que sí difiere es que `cuantiles` produce una rejilla **no uniforme**, y que
`layernorm` recalcula en vez de congelar.

**Congelar estadísticas es peligroso en este problema.** La desviación por canal de lo que
entra al bloque de salida es de 0.001 a 0.015: dividir por ella amplifica por setecientos, y
basta con que el tronco desplace la media de un canal en 0.01 para que la entrada normalizada
se aleje diez desviaciones y la base se quede sin datos encima.

Y la respuesta no es la misma para las tres bases. La B-spline tiene soporte **compacto**: se
anula fuera de su rejilla en vez de decaer, de modo que una entrada fuera del último nudo no
deja ninguna derivada. El Chebyshev es global y su `tanh` previo ya gestiona el rango, así
que no necesita corrección.

In [ ]:
# Comparacion de los cuatro modos sin GPU ni FitSNAP. Segundos.
!cd {MARCO} && python prueba_variantes.py

# Y dentro del pipeline real, con los mismos descriptores para los cuatro:
!cd {MARCO} && python barrida.py --que modo --raiz {RAIZ} \
    --grupo LiF_64_kjpaw --arquitectura gaussiana --epocas 8

### Dónde se fijan los valores elegidos

Viven en `kan_lib.py` como `BATCH_SIZE`, `TASA_APRENDIZAJE` y el modo por omisión. Hay dos
formas de cambiarlos, y conviene la primera:

1. **Editar `kan_lib.py`.** Queda escrito en el archivo y lo heredan el notebook y todos los
   trabajadores. Es lo que corresponde para la decisión definitiva.
2. **Variables de entorno**, para una prueba puntual: `KAN_BATCH_SIZE`, `KAN_LR`,
   `KAN_FRACCION`, `KAN_NORMALIZAR`, `KAN_RESIDUO`, `KAN_REJILLA_CADA`.

Lo que no sirve es cambiar una variable en una celda: los trabajadores son procesos aparte y
no ven el estado del notebook. `resumen_kan()` imprime la configuración efectiva, y conviene
llamarla al principio de cada corrida.

---

## 12. El presupuesto se cuenta en pasos de optimizador

Esta es la sección que gobierna a todas las demás.

El 2 de octubre se corrieron veinte modelos a 70 épocas, con nueve tasas entre `5e-6` y
`2e-1`, tres bases y cuatro modos de rango. **Ninguno aprendió las fuerzas.** Las variantes
KAN colapsaron a predecir fuerza cero y energía constante; las MLP produjeron fuerzas entre
cinco y diez veces peores que predecir cero.

La causa no es de arquitectura ni de hiperparámetros:

| | Piloto | Referencia del grupo |
|---|---|---|
| Configuraciones de entrenamiento | 664 | ~25 223 |
| Tamaño de lote | 50 | 50 |
| Pasos por época | 14 | ~505 |
| Épocas | 70 | 70 |
| **Pasos de optimizador** | **~930** | **~35 300** |
| Proporción | **2.6 %** | 100 % |

`FRACCION = 0.25` y el grupo único se pusieron para abaratar el piloto. Lo que hicieron fue
convertirlo en una medición de inicializaciones.

**La regla:** el presupuesto de una campaña se declara en pasos de optimizador, se elige para
igualar al menos los de la referencia, y las épocas son la consecuencia.

In [ ]:
# Pasos de optimizador de una corrida: la unica cuenta que decide si una
# campana puede medir algo.
import math

OBJETIVO_PASOS = 35_300          # los de la configuracion de referencia del grupo

def presupuesto(n_config, lote=None, epocas=None, objetivo=OBJETIVO_PASOS, nombre=""):
    lote = lote or BATCH_SIZE
    pasos_epoca = math.ceil(n_config / lote)
    pasos = pasos_epoca * (epocas or 0)
    razon = pasos / objetivo if objetivo else float("nan")
    epocas_nec = math.ceil(objetivo / pasos_epoca)
    print(f"{nombre or 'corrida':26s} {n_config:7d} cfg  lote {lote:4d}  "
          f"{pasos_epoca:5d} pasos/epoca  x {epocas or 0:4d} epocas = "
          f"{pasos:7d} pasos  ({razon*100:6.2f} % del objetivo)")
    if pasos < objetivo:
        print(f"{'':26s} para alcanzarlo hacen falta {epocas_nec} epocas")
    return dict(n_config=n_config, lote=lote, pasos_epoca=pasos_epoca,
                pasos=pasos, razon=razon, epocas_necesarias=epocas_nec)

def contar_configuraciones(raiz_datos=None, grupos=None, fraccion=None):
    raiz_datos = Path(raiz_datos or (carpeta_datos(RAIZ)))
    fraccion = FRACCION if fraccion is None else fraccion
    total = 0
    for d in sorted(raiz_datos.iterdir()):
        if d.is_dir() and (grupos is None or d.name in grupos):
            total += len(list(d.glob("*.json")))
    return int(total * fraccion * 0.7)      # 70 % entrenamiento, 30 % prueba

print(f"{'escenario':26s} {'cfg':>7s}")
print("-" * 104)
P_PILOTO = presupuesto(contar_configuraciones(grupos=["LiF_64_kjpaw"]),
                       epocas=70, nombre="piloto, 1 grupo, 25 %")
presupuesto(contar_configuraciones(fraccion=1.0), epocas=70,
            nombre="todos los grupos, 100 %")
print()
EPOCAS_OBJETIVO = P_PILOTO["epocas_necesarias"]
print(f"EPOCAS_OBJETIVO para el piloto = {EPOCAS_OBJETIVO}")
print()
print("Dos palancas: numero de configuraciones y epocas. Subir FRACCION cuesta")
print("descriptores (~7 min por trabajador y por cuarto); subir epocas cuesta")
print("~18 s cada una. Suele salir mas barato subir epocas, pero mas datos")
print("ademas reducen el sobreajuste: no son equivalentes.")

---

## 13. El criterio de medida es el error físico

El 2 de octubre siete configuraciones distintas —gaussiana, B-spline, Chebyshev y **MLP**,
con y sin normalización, a dos tasas y con dos semillas— terminaron todas con la misma
pérdida, entre `5.353e-02` y `5.464e-02`.

Siete arquitecturas no coinciden en dos cifras significativas por mérito propio. Al abrir
las predicciones apareció la explicación:

```
Fx_Truth -0.18334539811              Fx_Pred -1.4739326891e-06
Energy_Truth -508.04413157857965     Energy_Pred -507.7420109199615
Energy_Truth -508.04417146026720     Energy_Pred -507.7420109198456
```

Fuerza cero y energía constante: dos configuraciones distintas con la misma predicción hasta
el décimo decimal. Ese valor de pérdida es **la media del cuadrado de las fuerzas
verdaderas** —una propiedad del conjunto de datos, no de ningún modelo—, y por eso todas
caen en él.

| Columna de `mae.py` | Qué dice |
|---|---|
| `F MAE` | error de fuerza en eV/Å. La referencia del grupo está en **1.108e-02** |
| `E MAE` | error de energía en eV/átomo. La referencia está en **2.790e-03** |
| `F/cero` | el error dividido por el que daría predecir cero. **1.000 = no aprendió nada** |
| `sE/sE` | desviación de la energía predicha sobre la verdadera. **0 = predice una constante** |

**Una trampa que conviene conocer.** En `LiF_64_kjpaw` la energía total por átomo apenas
varía entre configuraciones, de modo que predecir una constante ya da un `E MAE` excelente;
algunas corridas degeneradas salieron por debajo de la referencia. Toda la señal está en las
fuerzas. Leer `E MAE` sin mirar `F/cero` lleva a conclusiones invertidas.

La pérdida de entrenamiento no se publica. El MAE sí.

In [ ]:
# El criterio, sobre cualquier carpeta de corridas.
!cd {MARCO} && python mae.py "{RAIZ}/corrida_*"

# Y la tabla de curvas, que marca sola las inservibles:
#   DIVERGIO: perdida inicial por encima de 1e6
#   SUELO:    perdida final en la franja del 5.4e-2
!cd {MARCO} && python tabla70.py {RAIZ}

---

## 14. Qué es exactamente «un KAN»: la ablación

Liu et al. (2024), en los detalles de implementación, declara dos mecanismos como necesarios
para que un KAN entrene:

$$\varphi(x) = w_b\, \mathrm{silu}(x) + \mathrm{spline}(x)$$

y la **actualización de la rejilla sobre la marcha**, que el propio paper motiva así: *«los
splines están definidos en regiones acotadas pero los valores de activación pueden
evolucionar»*. Medido el 2 de octubre: la entrada al bloque de salida pasa de 0.73 a 2 531 en
dos épocas.

Los dos están implementados y **apagados por omisión**, para que las condiciones ya medidas
sigan dando lo mismo:

| Variable | Qué hace |
|---|---|
| `KAN_RESIDUO=1` | añade el término residual. Cuesta `n_in × n_out` por capa KAN: **128 parámetros** en la condición principal |
| `KAN_REJILLA_CADA=200` | recalcula la rejilla a los cuantiles cada 200 pasos de entrenamiento |

### Las cuatro filas

| Fila | Configuración | Qué aísla |
|---|---|---|
| 1 | `1_mlp_base` | la referencia |
| 2 | `3_kan_gauss` sin residuo ni rejilla | el KAN mínimo |
| 3 | `3_kan_gauss` + `KAN_RESIDUO=1` | qué aporta el camino lineal |
| 4 | `3_kan_gauss` + residuo + rejilla adaptativa | el KAN completo del paper |

Sin las cuatro filas, un resultado negativo no dice «el KAN no ayuda en el bloque de salida
de un potencial SNAP»; dice «esta versión del KAN no ayuda». Es una diferencia que cualquier
revisor señala.

**Advertencia de atribución.** En esta posición —una sola capa KAN de 64→1 al final— el
término residual es literalmente `Linear(64→1)` aplicado sobre `silu(x)`, es decir, casi la
última capa del MLP. Si la fila 3 funciona y la 2 no, lo más probable es que el mérito sea
del camino lineal. Por eso la ablación es el experimento y no un añadido.

Queda abierta la decisión de si el bloque de salida son los 65 parámetros de la última capa o
los 4 225 de la cabeza de dos capas. Con una sola capa el residuo domina y el spline tiene
poco margen; con dos, el spline tiene dónde aportar.

In [ ]:
# Una fila por proceso: las variables de entorno se leen al importar kan_lib,
# asi que no se pueden mezclar dos configuraciones dentro del mismo proceso.
ABLACION = [
    ("1_mlp_base",     "linear",    {}),
    ("2_kan_base",     "gaussiana", {}),
    ("3_kan_resid",    "gaussiana", {"KAN_RESIDUO": "1"}),
    ("4_kan_completo", "gaussiana", {"KAN_RESIDUO": "1", "KAN_REJILLA_CADA": "200"}),
]

print(f"{'fila':16s} {'salida':11s} {'entorno':42s} {'parametros':>11s}")
print("-" * 86)
import subprocess as _sp
for _id, _sal, _env in ABLACION:
    _codigo = (f"import sys; sys.path.insert(0,'{MARCO}');"
               "from kan_lib import crear_red,n_params;"
               "print(n_params(crear_red([440,256,128,64,64,1],%r)))" % _sal)
    _n = _sp.run([sys.executable, "-c", _codigo], env=dict(os.environ, **_env),
                 capture_output=True, text=True).stdout.strip()
    _txt = " ".join(f"{k}={v}" for k, v in _env.items()) or "(ninguno)"
    print(f"{_id:16s} {_sal:11s} {_txt:42s} {_n:>11s}")

print()
print("El residuo mueve el conteo, asi que la condicion isoparametrica")
print("(7_kan_iso) hay que rehacerla antes de usar la ablacion como control")
print("de capacidad.")

---

## 15. Cuánto influye el KAN, más allá del conteo de parámetros

El 0.127 % dice cuánto **ocupa** el bloque KAN, no cuánto **influye**. Son preguntas
distintas. Tres medidas atacan la segunda, y ninguna requiere reentrenar:

| Medida | Qué responde |
|---|---|
| Ablación por linealización | si se sustituye cada función aprendida por su mejor recta, ¿cuánto cambia la energía predicha? |
| No linealidad efectiva | ¿qué fracción de cada función univariada **no** explica una recta? |
| Desplazamiento de parámetros | ¿se movieron los 448 durante el entrenamiento, o siguen donde empezaron? |

La primera necesita saber qué valores entran de verdad a la última capa, así que engancha
configuraciones reales en vez de suponer el rango.

**Dos trampas al leer estos números**, verificadas en simulación antes de escribir la celda:

1. **La ablación se lee en porcentaje, no en valor absoluto.** Un KAN forzado a ser lineal
   dio un cambio absoluto *mayor* que uno curvo, porque sus pesos eran de mayor escala. En
   relativo la distinción es limpia: 0.3 % contra 62 %.
2. **La no linealidad tiene un suelo.** Ocho gaussianas no pueden reproducir una recta
   exacta; el mínimo alcanzable resultó ~3×10⁻³. La celda imprime ese suelo y la razón contra
   él. Sin esa referencia, un 3×10⁻³ se leería como «algo de no linealidad» cuando significa
   lo contrario.

**La medida principal sigue siendo la propia rejilla.** La influencia del KAN es la
diferencia de MAE entre condiciones que solo difieren en esa capa: 1↔3 (con 448 parámetros
de ventaja), 1↔7 (±10) y 8↔3 (±14). Si las tres apuntan igual, el efecto es real. Estas tres
medidas dicen *por qué*, no *si*.

In [ ]:
# Sobre un modelo YA entrenado con KAN. Requiere haber corrido antes la prueba
# corta con salida_kan="gaussiana", o cargar un modelo guardado.
#
# INFLU = medir_influencia_kan(r["fs"])
# dibujar_funciones_kan(INFLU, n=12, archivo=RAIZ / "funciones_kan.png")

print("Celda de analisis. Descomentar despues de una corrida con KAN.")
print("medir_influencia_kan y dibujar_funciones_kan estan en kan_lib.")

---

## 16. El piloto y el lanzador multi-GPU

### El piloto

Las 8 condiciones × 5 semillas sobre **un solo grupo** en vez de los doce. No es un recorte
del diseño: son las mismas condiciones, las mismas semillas y el mismo presupuesto. Lo único
reducido es el conjunto de datos — y por eso hay que fijar las épocas con la sección 14, no
con el número heredado.

### El lanzador

FitSNAP usa **una** GPU. Si se alquilan ocho, siete están apagadas mientras se pagan.
`lanzar.py` reparte las corridas entre las tarjetas que detecte, cada trabajador es un
proceso atado a su tarjeta con `CUDA_VISIBLE_DEVICES`, y cada uno calcula los descriptores
**una vez** y encadena sus corridas cambiando la arquitectura en caliente.

### Lo que cuesta, medido

| Magnitud | 2× RTX 4090 | 8× RTX 3090 |
|---|---|---|
| Descriptores | 452–455 s con 2 trabajadores | **414–425 s con 8** |
| Segundos por época | 17.8 | **18.2** |
| `error_analysis` por corrida | ~22 s | ~30 s |
| RAM por trabajador | 29.5 GiB entrenando, 35.6 de pico | 145 GB agregados con 8 |
| VRAM por trabajador, lote 50 | 5 678 MiB | 5 743 MiB |
| CPU durante descriptores | 3–4 núcleos de 64 | — |

Dos lecturas para dimensionar el alquiler. **El disco no sufre con ocho trabajadores:** los
25 223 archivos JSON sueltos se leen por ocho procesos a la vez más rápido que por dos.
Y **la tarjeta casi no importa:** la 3090 cuesta un 2 % por época frente a la 4090,
exactamente lo que predice el 5 % de utilización de GPU medido durante el ajuste.

Lo que sí limita es el presupuesto de pasos. Multiplicar trabajadores acorta el reloj de la
rejilla pero no cambia cuántas veces se actualizan los pesos en cada corrida: ocho tarjetas
producen cuarenta modelos sin entrenar más rápido que dos.

**Antes de lanzar**, mirar que la columna `delta` del trabajador sea mayor que cero. Si es
cero, los pesos no se movieron y no hay resultado, aunque `metrics.dat` exista.

In [ ]:
# Sonda: UNA corrida, un trabajador, con el monitor por fases. En una maquina
# nueva da los segundos por epoca, la VRAM y la RAM reales de un trabajador.
!cd {MARCO} && python monitor_fases.py \
    --csv {RAIZ}/sonda.csv -- \
    python trabajador.py \
    '[{{"id":"1_mlp_base","capas":"num_desc 256 128 64 64 1","salida":"linear","kw":{{}},"semilla":1111}}]' \
    {RAIZ} LiF_64_kjpaw 5 {RAIZ}/sonda 0

### Validación de punta a punta

Las 8 condiciones con una semilla y pocas épocas. Comprueba la maquinaria, no las
arquitecturas.

In [ ]:
!cd {MARCO} && python lanzar.py \
    --raiz {RAIZ} --semillas 1111 --epocas 3 --si

# El criterio no es la perdida:
!cd {MARCO} && python mae.py "{RAIZ}/corrida_*"

### La rejilla completa

Con el presupuesto de la sección 14, no con un número heredado.

In [ ]:
# Descomentar cuando la validacion haya pasado y EPOCAS_OBJETIVO este calculado.
#
# !cd {MARCO} && python lanzar.py \
#     --raiz {RAIZ} --epocas {EPOCAS_OBJETIVO} --trabajadores 8 --si

print(f"Listo para lanzar con --epocas {EPOCAS_OBJETIVO}.")
print("Reloj estimado por corrida: "
      f"{7 + EPOCAS_OBJETIVO * 18.2 / 60:.0f} min "
      "(7 de descriptores + 18.2 s por epoca).")

### Recoger los resultados del piloto

In [ ]:
# Junta los metrics.dat de todos los trabajadores de una carpeta de corrida.
import json as _json, glob as _glob

CARPETA = sorted(RAIZ.glob("corrida_*"))[-1] if list(RAIZ.glob("corrida_*")) else None
print("carpeta:", CARPETA)

filas, fallos, sin_entrenar = [], [], []
if CARPETA:
    for _reg in sorted(CARPETA.glob("trabajador_*.jsonl")):
        for _l in _reg.read_text().splitlines():
            if not _l.strip():
                continue
            _d = _json.loads(_l)
            if _d.get("evento") != "corrida":
                continue
            if not _d.get("ok"):
                fallos.append(_d)
            elif not _d.get("entreno", True):
                sin_entrenar.append(_d)
            else:
                try:
                    _m = leer_metrics(Path(_d["metrics"]))
                    _m["condicion"] = _d["id"]; _m["semilla"] = _d["semilla"]
                    _m["min"] = _d["min"]
                    filas.append(_m)
                except Exception as _e:
                    fallos.append(dict(_d, error=f"metrics ilegible: {_e}"))

print(f"con resultado y pesos movidos : {len(filas)}")
print(f"sin mover los pesos           : {len(sin_entrenar)}")
print(f"fallidas                      : {len(fallos)}")
for _f in fallos[:6]:
    print(f"  {_f.get('id')} s{_f.get('semilla')}: {str(_f.get('error'))[:110]}")

if filas:
    TABLA = pd.concat(filas, ignore_index=True)
    TABLA.to_csv(RAIZ / "resultados.csv", index=False)
    print()
    print(f"guardado en {RAIZ / 'resultados.csv'}")
    print(TABLA.head(20).to_string())

---

## 17. Despliegue en clúster

En un clúster con planificador no se corre un notebook: se envían trabajos. El marco ya está
pensado para eso —cada trabajador es un proceso independiente—, así que basta envolverlo.

**Lo que hay que rellenar** está marcado con `AJUSTAR`: la cuenta, la partición, el módulo o
entorno que carga Python y el tiempo máximo. Eso cambia de un clúster a otro.

**El patrón es un arreglo de trabajos**, una condición y semilla por tarea, en vez de un solo
trabajo que las encadene. Si una tarea falla no se lleva las demás, el planificador las
reparte conforme haya hueco, y se puede reenviar una sola.

In [ ]:
# Escribe el guion de lote. NO lo envia: revisar antes las lineas AJUSTAR.
PLANTILLA = """#!/bin/bash
#SBATCH --job-name=kan-snap
#SBATCH --account=AJUSTAR_CUENTA
#SBATCH --partition=AJUSTAR_PARTICION
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=8
#SBATCH --gpus=1
#SBATCH --time=AJUSTAR_HH:MM:SS
#SBATCH --array=0-{n_tareas}
#SBATCH --output=logs/kan_%A_%a.out
#SBATCH --error=logs/kan_%A_%a.err

set -euo pipefail
mkdir -p logs

# --- entorno. AJUSTAR: modulos o conda, segun el cluster -------------------
# module load cuda/12.4
# source /ruta/al/conda/etc/profile.d/conda.sh && conda activate fitsnap
PY=${{PY:-python}}

RAIZ={raiz}
cd "$RAIZ/MARCO_KAN"

COND=({condiciones})
SEMILLAS=({semillas})
N_SEM=${{#SEMILLAS[@]}}
I=$SLURM_ARRAY_TASK_ID
C=${{COND[$((I / N_SEM))]}}
S=${{SEMILLAS[$((I % N_SEM))]}}

echo "tarea $I: condicion $C, semilla $S"
$PY -c "import sys; sys.path.insert(0,'.'); from kan_lib import resumen_kan; print(resumen_kan())"

$PY lanzar.py --raiz "$RAIZ" --condiciones "$C" --semillas "$S" \\
    --epocas {epocas} --trabajadores 1 \\
    --salida "$RAIZ/campana_$SLURM_ARRAY_JOB_ID" --si
"""

_cond = [c[0] for c in CONDICIONES]
_sem = list(SEMILLAS)
_n = len(_cond) * len(_sem) - 1

_guion = PLANTILLA.format(n_tareas=_n, raiz=RAIZ,
                          condiciones=" ".join(_cond),
                          semillas=" ".join(str(s) for s in _sem),
                          epocas=EPOCAS_OBJETIVO)
_ruta = RAIZ / "enviar_kan.sbatch"
_ruta.write_text(_guion, encoding="utf-8")
print(f"escrito {_ruta}  ({_n + 1} tareas)")
print()
print(_guion)

In [ ]:
# Envio y seguimiento. Descomentar cuando las lineas AJUSTAR esten puestas.
#
# !sbatch {RAIZ}/enviar_kan.sbatch
# !squeue -u $USER
# !cd {MARCO} && python mae.py "{RAIZ}/campana_*"

print("Revisar las lineas AJUSTAR antes de enviar.")
print()
print("Comprobaciones minimas antes de ocupar un nodo, en orden:")
print("  1. python verificar_lib.py kan_lib.py trabajador.py barrida.py")
print("  2. python prueba_variantes.py          (segundos, sin GPU)")
print("  3. python lanzar.py --solo-comprobar")
print("  4. una tarea del arreglo a pocas epocas, y mirar mae.py")
print("  5. recien entonces el arreglo completo")

## 18. Antes de destruir la instancia

In [ ]:
!tar czf {RAIZ}/resultados_kan.tar.gz -C {RAIZ} resultados
!ls -lh {RAIZ}/resultados_kan.tar.gz
# Bajarlo con:  scp -P <PUERTO> root@<HOST>:{RAIZ}/resultados_kan.tar.gz .

---

## Lo establecido, y lo que falta

**Medido, y no hace falta volver a medirlo:**

- El suelo degenerado de este conjunto vale **5.4e-02** y es la media del cuadrado de las
  fuerzas verdaderas. Es del conjunto de datos, no de ningún modelo.
- El umbral de divergencia del pipeline está **entre 2e-2 y 5e-2**.
- La B-spline recibe gradiente **exactamente cero** fuera de su rejilla; el Chebyshev no
  necesita corrección de rango.
- Las activaciones del tronco crecen de 0.73 a 2 531 en dos épocas.
- La capa KAN **sí entrena**: 98.16 % de caída en 70 épocas con normalización por muestra.
- Ocho trabajadores simultáneos no producen contención de disco ni de RAM.
- `dtype_setting` está fuera de la lista blanca de esta versión de FitSNAP: float64 no tiene
  vía por aquí.

**Lo que falta, en orden:**

| Pendiente | Por qué bloquea |
|---|---|
| Igualar el presupuesto de pasos de la referencia | Sin esto ninguna comparación de arquitecturas es interpretable |
| Reproducir el modelo de referencia con este marco | Si no llega a 1.108e-02 eV/Å, el problema no es la arquitectura |
| Correr la ablación de cuatro filas | Sin ella, un resultado negativo describe esta versión del KAN, no el KAN |
| Rehacer la condición isoparamétrica con el residuo | El residuo añade 128 parámetros y mueve el control de capacidad |
| Decidir si el bloque de salida son 65 o 4 225 parámetros | Con una sola capa el residuo domina y el spline queda sin margen |
| Comprobar qué incluye la implementación de Wang et al. | Si su KAN tampoco tiene residuo, su resultado cobra otro sentido |
| Medir ms por paso de MD en LAMMPS | Requiere desplegar el potencial vía ML-IAP |